# 胚胎发育中的细胞形状分析 (CMap 数据版)

## 3DCSQ + EmbSAM: 三维细胞形状量化分析系统

本 Jupyter Notebook 专门用于分析 CMap 数据集中的胚胎细胞形状

In [18]:
# 导入必要的库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
from pathlib import Path
import glob
import nibabel as nib
from scipy import ndimage
from scipy.spatial import ConvexHull
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pyshtools as pysh

# 导入项目自定义模块
import sys
sys.path.append('.')

import utils.cell_func as cell_f
import utils.general_func as general_f
import transformation.SH_represention as sh_represent
from analysis.SH_analyses import analysis_SHcPCA_One_embryo, analysis_SHc_Kmeans_One_embryo

# 设置 matplotlib 显示中文
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print("✓ 库导入成功")

✓ 库导入成功


### 配置 CMap 数据路径

In [24]:
# 配置基础数据路径
BASE_DATA_PATH = Path(r"E:\CityU-Onedrive-Backup\Zelin OneDrive - City University of Hong Kong - Student\MembraneProjectData\CMapSubmission\Dataset Access")

# 数据集路径
DATASET_C_PATH = BASE_DATA_PATH / "Dataset C"
DATASET_G_PATH = BASE_DATA_PATH / "Dataset G"

# 输出路径
OUTPUT_PATH = Path("./results")
OUTPUT_PATH.mkdir(exist_ok=True)

print("=" * 60)
print("数据路径配置")
print("=" * 60)
print(f"基础数据路径：{BASE_DATA_PATH}")
print(f"Dataset C 路径：{DATASET_C_PATH}")
print(f"Dataset G 路径：{DATASET_G_PATH}")
print(f"输出路径：{OUTPUT_PATH}")
print("=" * 60)

# 验证路径是否存在
if not BASE_DATA_PATH.exists():
    print(f"\n❌ 错误：基础数据路径不存在！\n{BASE_DATA_PATH}")
    print("\n请检查路径是否正确。")
else:
    print("\n✓ 基础数据路径存在")
    
    # 列出可用的数据集
    available_datasets = []
    if DATASET_C_PATH.exists():
        available_datasets.append("Dataset C")
        print(f"✓ Dataset C 可用")
    if DATASET_G_PATH.exists():
        available_datasets.append("Dataset G")
        print(f"✓ Dataset G 可用")
    
    if not available_datasets:
        print("\n⚠️  警告：没有找到可用的数据集！")

数据路径配置
基础数据路径：E:\CityU-Onedrive-Backup\Zelin OneDrive - City University of Hong Kong - Student\MembraneProjectData\CMapSubmission\Dataset Access
Dataset C 路径：E:\CityU-Onedrive-Backup\Zelin OneDrive - City University of Hong Kong - Student\MembraneProjectData\CMapSubmission\Dataset Access\Dataset C
Dataset G 路径：E:\CityU-Onedrive-Backup\Zelin OneDrive - City University of Hong Kong - Student\MembraneProjectData\CMapSubmission\Dataset Access\Dataset G
输出路径：results

✓ 基础数据路径存在
✓ Dataset C 可用
✓ Dataset G 可用


### 扫描胚胎样本

In [29]:
def scan_embryo_samples(dataset_path, dataset_name):
    """
    扫描数据集中的所有胚胎样本
    """
    embryo_dict = {}
    
    if not dataset_path.exists():
        return embryo_dict
    
    # 查找所有胚胎目录
    embryo_dirs = [d for d in dataset_path.iterdir() if d.is_dir()]
    
    print(f"\n{dataset_name} - 找到 {len(embryo_dirs)} 个胚胎样本:")
    
    for embryo_dir in sorted(embryo_dirs):
        embryo_name = embryo_dir.name
        
        # 查找 SegCell 子目录
        segcell_dir = embryo_dir / "SegCell"
        
        if segcell_dir.exists():
            # 在 SegCell 目录中查找所有时间点文件
            timepoint_files = sorted(glob.glob(str(segcell_dir / "*_segCell.nii.gz")))
            
            if timepoint_files:
                embryo_dict[embryo_name] = {
                    'path': embryo_dir,
                    'segcell_path': segcell_dir,
                    'timepoints': timepoint_files,
                    'num_timepoints': len(timepoint_files)
                }
                print(f"  • {embryo_name}: {len(timepoint_files)} 个时间点")
        else:
            print(f"  ⚠️  {embryo_name}: 没有找到 SegCell 目录")
    
    return embryo_dict


# 扫描所有数据集
all_embryos = {}

if DATASET_C_PATH.exists():
    embryos_c = scan_embryo_samples(DATASET_C_PATH, "Dataset C")
    all_embryos['Dataset C'] = embryos_c

if DATASET_G_PATH.exists():
    embryos_g = scan_embryo_samples(DATASET_G_PATH, "Dataset G")
    all_embryos['Dataset G'] = embryos_g

# 统计信息
print("\n" + "=" * 60)
print("胚胎样本统计")
print("=" * 60)
total_embryos = 0
total_timepoints = 0

for dataset_name, embryos in all_embryos.items():
    num_embryos = len(embryos)
    num_timepoints = sum([info['num_timepoints'] for info in embryos.values()])
    total_embryos += num_embryos
    total_timepoints += num_timepoints
    print(f"{dataset_name}: {num_embryos} 个胚胎，{num_timepoints} 个时间点")

print(f"\n总计：{total_embryos} 个胚胎，{total_timepoints} 个时间点")
print("=" * 60)


Dataset C - 找到 12 个胚胎样本:
  • MT_lag-1_Sample1: 195 个时间点
  • MT_lag-1_Sample2: 195 个时间点
  • MT_pop-1_Sample1: 140 个时间点
  • MT_pop-1_Sample2: 155 个时间点
  • WT_Sample1: 255 个时间点
  • WT_Sample2: 195 个时间点
  • WT_Sample3: 185 个时间点
  • WT_Sample4: 220 个时间点
  • WT_Sample5: 195 个时间点
  • WT_Sample6: 205 个时间点
  • WT_Sample7: 205 个时间点
  • WT_Sample8: 195 个时间点

Dataset G - 找到 17 个胚胎样本:
  • Sample04: 150 个时间点
  • Sample05: 170 个时间点
  • Sample06: 210 个时间点
  • Sample07: 165 个时间点
  • Sample08: 160 个时间点
  • Sample09: 160 个时间点
  • Sample10: 160 个时间点
  • Sample11: 170 个时间点
  • Sample12: 165 个时间点
  • Sample13: 150 个时间点
  • Sample14: 155 个时间点
  • Sample15: 170 个时间点
  • Sample16: 160 个时间点
  • Sample17: 160 个时间点
  • Sample18: 160 个时间点
  • Sample19: 160 个时间点
  • Sample20: 170 个时间点

胚胎样本统计
Dataset C: 12 个胚胎，2340 个时间点
Dataset G: 17 个胚胎，2795 个时间点

总计：29 个胚胎，5135 个时间点


### 选择要分析的胚胎样本

In [33]:
# 选择要分析的胚胎
# 方法 1: 分析所有胚胎
# SELECTED_DATASET = "Dataset C"
# SELECTED_EMBRYOS = list(all_embryos.get(SELECTED_DATASET, {}).keys())

# 方法 2: 分析特定胚胎（推荐）
# Dataset C 中的胚胎列表 (共 12 个):
#   野生型 (8 个): WT_Sample1 (255tp), WT_Sample2 (195tp), WT_Sample3 (185tp), 
#                WT_Sample4 (220tp), WT_Sample5 (195tp), WT_Sample6 (205tp), 
#                WT_Sample7 (205tp), WT_Sample8 (195tp)
#   lag-1 突变体 (2 个): MT_lag-1_Sample1 (195tp), MT_lag-1_Sample2 (195tp)
#   pop-1 突变体 (2 个): MT_pop-1_Sample1 (140tp), MT_pop-1_Sample2 (155tp)
SELECTED_DATASET = "Dataset C"
SELECTED_EMBRYOS = ["WT_Sample1", "WT_Sample2", "WT_Sample3"]  # 修改为您想分析的胚胎

# 方法 3: 分析单个胚胎
# SELECTED_DATASET = "Dataset C"
# SELECTED_EMBRYOS = ["Sample04LabelUnified"]

print(f"选择的数据集：{SELECTED_DATASET}")
print(f"选择的胚胎：{SELECTED_EMBRYOS}")

# 验证选择的胚胎是否存在
available_embryos = list(all_embryos.get(SELECTED_DATASET, {}).keys())
valid_embryos = [e for e in SELECTED_EMBRYOS if e in available_embryos]

if not valid_embryos:
    print(f"\n❌ 错误：选择的胚胎在 {SELECTED_DATASET} 中不存在！")
    print(f"可用的胚胎：{available_embryos}")
else:
    if len(valid_embryos) < len(SELECTED_EMBRYOS):
        invalid = set(SELECTED_EMBRYOS) - set(valid_embryos)
        print(f"\n⚠️  以下胚胎不存在：{invalid}")
        print(f"可用的胚胎：{available_embryos}")
    
    print(f"\n✓ 将分析 {len(valid_embryos)} 个胚胎")
    SELECTED_EMBRYOS = valid_embryos

选择的数据集：Dataset C
选择的胚胎：['WT_Sample1', 'WT_Sample2', 'WT_Sample3']

✓ 将分析 3 个胚胎


### 加载细胞名称字典

In [36]:
# 加载细胞名称映射
NAME_DICT_PATH = os.path.join(BASE_DATA_PATH,SELECTED_DATASET,"name_dictionary.csv")

try:
    label_name_dict, name_label_dict = cell_f.get_cell_name_affine_table(str(NAME_DICT_PATH))
    print("✓ 细胞名称字典加载成功")
    print(f"  字典路径：{NAME_DICT_PATH}")
    print(f"  细胞数量：{len(label_name_dict)}")
except Exception as e:
    print(f"⚠️  无法加载细胞名称字典：{e}")
    print(f"  尝试使用备用路径...")
    try:
        alt_path = r'D:\cell_shape_quantification\DATA\name_dictionary_no_name.csv'
        label_name_dict, name_label_dict = cell_f.get_cell_name_affine_table(alt_path)
        print("✓ 从备用路径加载成功")
    except:
        label_name_dict = {}
        name_label_dict = {}
        print("❌ 无法加载细胞名称字典，将使用默认命名")

✓ 细胞名称字典加载成功
  字典路径：E:\CityU-Onedrive-Backup\Zelin OneDrive - City University of Hong Kong - Student\MembraneProjectData\CMapSubmission\Dataset Access\Dataset C\name_dictionary.csv
  细胞数量：1362


## 2. 细胞表面提取与分析

In [38]:
# 存储所有分析结果
analysis_results = {}

if valid_embryos:
    print(f"开始分析 {len(valid_embryos)} 个胚胎...")
    print("=" * 60)
    
    for embryo_idx, embryo_name in enumerate(valid_embryos, 1):
        print(f"\n{'='*60}")
        print(f"分析胚胎 {embryo_idx}/{len(valid_embryos)}: {embryo_name}")
        print(f"{'='*60}")
        
        # 获取胚胎信息
        embryo_info = all_embryos[SELECTED_DATASET][embryo_name]
        embryo_path = embryo_info['path']
        segcell_path = embryo_info.get('segcell_path', embryo_path)
        timepoint_files = embryo_info['timepoints']
        
        print(f"胚胎目录：{embryo_path}")
        print(f"SegCell 路径：{segcell_path}")
        print(f"时间点数量：{len(timepoint_files)}")
        
        # 初始化该胚胎的结果存储
        analysis_results[embryo_name] = {
            'dataset': SELECTED_DATASET,
            'timepoints': {},
            'stats': []
        }
        
        # 选择第一个时间点作为示例分析
        selected_timepoint_file = timepoint_files[0] if timepoint_files else None
        
        if selected_timepoint_file:
            print(f"\n分析时间点：{Path(selected_timepoint_file).name}")
            
            # 加载胚胎数据
            img = nib.load(selected_timepoint_file)
            img_data = img.get_fdata().astype(np.int16)
            
            print(f"数据形状：{img_data.shape}")
            print(f"数据值范围：{img_data.min()} - {img_data.max()}")
            
            # 获取所有细胞标签
            cell_labels = np.unique(img_data)
            cell_labels = cell_labels[cell_labels != 0]  # 去除背景
            print(f"检测到 {len(cell_labels)} 个细胞")
            
            # 存储当前分析状态
            current_embryo = embryo_name
            current_timepoint = Path(selected_timepoint_file).stem
            current_img_data = img_data
            current_cell_labels = cell_labels
            selected_timepoint = selected_timepoint_file
            
            print(f"\n✓ 数据加载成功")
        else:
            print(f"⚠️  胚胎 {embryo_name} 没有找到时间点文件")
            continue
else:
    print("❌ 没有选择有效的胚胎样本")

开始分析 3 个胚胎...

分析胚胎 1/3: WT_Sample1
胚胎目录：E:\CityU-Onedrive-Backup\Zelin OneDrive - City University of Hong Kong - Student\MembraneProjectData\CMapSubmission\Dataset Access\Dataset C\WT_Sample1
SegCell 路径：E:\CityU-Onedrive-Backup\Zelin OneDrive - City University of Hong Kong - Student\MembraneProjectData\CMapSubmission\Dataset Access\Dataset C\WT_Sample1\SegCell
时间点数量：255

分析时间点：WT_Sample1_001_segCell.nii.gz
数据形状：(256, 356, 214)
数据值范围：0 - 1209
检测到 2 个细胞

✓ 数据加载成功

分析胚胎 2/3: WT_Sample2
胚胎目录：E:\CityU-Onedrive-Backup\Zelin OneDrive - City University of Hong Kong - Student\MembraneProjectData\CMapSubmission\Dataset Access\Dataset C\WT_Sample2
SegCell 路径：E:\CityU-Onedrive-Backup\Zelin OneDrive - City University of Hong Kong - Student\MembraneProjectData\CMapSubmission\Dataset Access\Dataset C\WT_Sample2\SegCell
时间点数量：195

分析时间点：WT_Sample2_001_segCell.nii.gz
数据形状：(256, 356, 214)
数据值范围：0 - 1210
检测到 4 个细胞

✓ 数据加载成功

分析胚胎 3/3: WT_Sample3
胚胎目录：E:\CityU-Onedrive-Backup\Zelin OneDrive - City Univer